## Simple GEN AI App using Langchain

### What we do?
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB

In [1]:
# Export all the variables inside .env
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
# Langsmith tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
# Data Ingestion-- From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader
loader= WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial#traces")




USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs=loader.load()
docs


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#traces', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialEngineChatTracing setupIntegrationsManual instrumentationMessages viewThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroublesho

In [4]:
# split the docs into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_spitter= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents= text_spitter.split_documents(docs)


In [5]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#traces', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialEngineChatTracing setupIntegrationsManual instrumentationMessages viewThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroublesho

In [6]:
# Embedding
from langchain_openai import OpenAIEmbeddings
embeddings= OpenAIEmbeddings()



In [7]:
# FAISS DB(It is created by FaceBook)---> For storing Vectors
from langchain_community.vectorstores import FAISS
vector_storeDB= FAISS.from_documents(documents,embeddings)



In [8]:
vector_storeDB

In [9]:
# Query from vector store DB

query= "Tracing the LLM call is useful, but tracing the full pipeline (including retrieval) gives you a complete overview of your application’s behavior."
result= vector_storeDB.similarity_search(query)
result[0].page_content

'Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialEngineChatTracing setupIntegrationsManual instrumentationMessages viewThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesView tracesFilter tracesConfigure run previewsManage a traceQuery tracesSDKQuery threadsSDKLangSmith Remote MCPLangSmith MCP ServerBulk export trace dataFind and fix issuesEngineEngine webhook eventsAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback 

In [13]:
# Create llm
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

### Retrievers Chains

 ##### Retrievals enable Large Language Model to use external data sources. LLMs only generate responses on their own based on training data which can be outdated or incomplete. Retrieval chains solve this limitation by linking LLMs to live, curated or private knowledge.



In [14]:
# Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
prompt= ChatPromptTemplate.from_template(
    """
    Answer the following question based on the provided context:
    <context>
    {context}
    </context>
    """
)
document_chain= create_stuff_documents_chain(llm, prompt)



In [15]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000134F8B365C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000134F8B37010>, root_client=<openai.OpenAI object at 0x00000134E7367F10>, root_async_client=<openai.AsyncOpenAI object at 0x00000134F8B36CB0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwarg

In [16]:
from langchain_core.documents import Document
document_chain.invoke(
    {
        "input": "Tracing the LLM call is useful, but tracing the full pipeline (including retrieval)",
        "context": [Document(page_content="Tracing the LLM call is useful, but tracing the full pipeline (including retrieval) gives you a complete overview of your application’s behavior. ")]
    }
)

"Why is tracing the full pipeline, including retrieval, important for understanding an application's behavior?\n\nTracing the full pipeline, including retrieval, is important because it provides a comprehensive view of the application's behavior. By examining each step in the process—not just the large language model (LLM) call but also the data retrieval and other components—you can identify where issues may be occurring, optimize performance, and ensure that all parts are working together as intended. This holistic understanding helps in debugging, improving efficiency, and designing more robust applications."

## However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.


## What is a Retriever in RAG?

#### In RAG pipelines, the retriever is the component responsible for fetching relevant pieces of information (documents, passages, chunks, or embeddings) from a knowledge base that the LLM (generator) can use to answer a query. It is nothing but a function which is taking a query in input from user and returning multiple document objects as output.

#### Without a retriever, the LLM answers only from its parametric memory (training data). With a retriever, it can access external non-parametric memory (vector stores, databases, APIs).

#### So retrievers bridge the gap between the user query and the knowledge base.
Knowledge base----> Any Vector store DB

In [17]:
# Create a retriever

retriever= vector_storeDB.as_retriever()

from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever, document_chain)


In [18]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000134F8A6D900>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
   

In [20]:
# Get response from the LLM
response= retrieval_chain.invoke({"input":"Tracing the LLM call is useful, but tracing the full pipeline (including retrieval)"})
response['answer']


'Based on the provided context, here is a brief overview relevant to tracing an LLM application with LangSmith:\n\n1. **Purpose of the Tutorial**: The tutorial guides you through building a customer support chatbot using retrieval-augmented generation (RAG) and enhancing the application with LangSmith observability throughout the development phases: prototyping, beta testing, and production.\n\n2. **Tracing and Observability**: The process involves tracing individual LLM calls and entire application pipelines, which helps in collecting and querying user feedback, logging metadata, and conducting A/B testing. It also involves using monitoring dashboards to track production performance.\n\n3. **Prerequisites**: Before starting, ensure you have:\n   - A LangSmith account.\n   - A LangSmith API key.\n   - An OpenAI API key.\n   - Optionally, install LangSmith CLI for terminal inspection of traces.\n   - Required Python and TypeScript packages (`langsmith` and `openai`).\n\n4. **Application

In [21]:
response

{'input': 'Tracing the LLM call is useful, but tracing the full pipeline (including retrieval)',
 'context': [Document(id='8e11f7f7-fa29-4d68-8f07-11ef57a03f8f', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#traces', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Remote MCPLangSmith MCP ServerBulk export trace dataFind and fix issuesEngineEngine webhook eventsAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback & evaluationManage evaluatorsLog user feedback using the SDKCollect feedback with presigned URLsSet up online evaluatorsMonitoring & alertingMonitor projects with dashboardsAlertsInsightsData type referenceRun (span) data formatFeedback data formatTrace query syntaxOn this pagePrerequisitesPrototypingSet up your environmentTrace LLM callsTrace

In [22]:
response['context']

[Document(id='8e11f7f7-fa29-4d68-8f07-11ef57a03f8f', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#traces', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Remote MCPLangSmith MCP ServerBulk export trace dataFind and fix issuesEngineEngine webhook eventsAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback & evaluationManage evaluatorsLog user feedback using the SDKCollect feedback with presigned URLsSet up online evaluatorsMonitoring & alertingMonitor projects with dashboardsAlertsInsightsData type referenceRun (span) data formatFeedback data formatTrace query syntaxOn this pagePrerequisitesPrototypingSet up your environmentTrace LLM callsTrace the whole pipelineCheck your traces from the terminalBeta testingCollect feedbackLog metadataProductionMonit